# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. All references to entities (record sets, fields, columns) use their unique `@id` as per the FAIR Croissant schema.

### Dataset Source
The dataset is hosted in a FAIR Croissant package and can be accessed via the schema URL below.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# mlcroissant metadata: access as object
metadata_obj = dataset.metadata
print("Dataset Title:", metadata_obj.name)
print("Description:", metadata_obj.description)
print("Identifier:", metadata_obj.identifier)
print("Authors (@id):", [author['@id'] for author in metadata_obj.author])
print("Keywords:", metadata_obj.keywords)
print("Date Published:", metadata_obj.datePublished)
print("License:", metadata_obj.license)


## 2. Data Overview
Review the available record sets, their `@id`s, and fields in the dataset. All references are made by `@id`.


In [ ]:
# Get record sets from the Croissant schema using dataset API
record_sets = list(dataset.record_sets())

# Print each record set and its fields using @id
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field['@id']}: {field.get('name')}")
        else:
            print(f"    - {field}")
    print()


## 3. Data Extraction
Load data from all available record sets into DataFrames. Use `@id` for referencing each record set and field.


In [ ]:
# Collect record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id} columns: {df.columns.tolist()}")
    print(df.head(2), "\n")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, and grouping by attributes. We'll demonstrate on the first record set (if numeric fields are present).


In [ ]:
from numpy import number

# Select a record set (@id)
selected_record_set_id = record_set_ids[0]
df = dataframes[selected_record_set_id]

# Find numeric fields by type or content
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in {selected_record_set_id}:", numeric_fields)

if numeric_fields:
    numeric_field = numeric_fields[0]
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records in {selected_record_set_id} with {numeric_field} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    categorical_fields = [col for col in df.columns if (df[col].dtype == object) and (col != numeric_field)]
    if categorical_fields:
        group_field = categorical_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:\n", grouped_df.head())


## 5. Visualization
Visualize distributions and relationships. We plot histograms of the selected numeric field and bar charts for grouped results if available.


In [ ]:
# Visualization of numeric fields
if numeric_fields:
    plt.figure(figsize=(6,3))
    df[numeric_field].hist(bins=5)
    plt.title(f"Distribution of {numeric_field} in {selected_record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Bar plot for grouped values
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(6,3))
        plt.bar(grouped_df[group_field], grouped_df[numeric_field])
        plt.title(f"Grouped Mean of {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()


## 6. Conclusion
We explored the FAIR^2 dataset using `mlcroissant`, inspecting record sets and fields referenced by their `@id`. Data was loaded, filtered, normalized, and visualized. This workflow demonstrates how to leverage Croissant schemas for reproducible, FAIR data science in biomedical research.

**Note:** Due to the dataset's small sample size, inferences are illustrative only. See the Croissant metadata for guidance on recommended use cases and limitations.